# Nyaya-Sahayak — Supreme Court Landmark Judgments Ingestion

Ingests curated landmark Supreme Court judgments into the `legal_rag_corpus`
Delta table with `doc_type="sc_judgment"` metadata for RAG retrieval.

We select ~50 key holdings from the most cited judgments across all legal
domains covered by the triage assistant. Each judgment is chunked into
structured holdings — not raw PDF text — so retrieval is precise.

## 1. Define Landmark Judgments

In [0]:
JUDGMENTS = [
    # -----------------------------------------------------------------------
    # CRIMINAL — Domestic Violence / Cruelty
    # -----------------------------------------------------------------------
    {
        "case_name": "Arnesh Kumar v. State of Bihar",
        "citation": "(2014) 8 SCC 273",
        "year": 2014,
        "domain": "criminal",
        "situation_types": ["domestic_violence", "criminal_general"],
        "applicable_sections": ["IPC Section 498A", "BNS Section 85"],
        "holdings": [
            "The Supreme Court directed that police should not automatically arrest the accused in cases under Section 498A IPC (now BNS Section 85). Arrest should be the last resort and not routine.",
            "Before making an arrest, the police officer must be satisfied that the arrest is necessary under the parameters laid down in Section 41 CrPC (now BNSS Section 35). The reasons must be recorded in writing.",
            "The Magistrate must not automatically authorise detention. The Magistrate must be satisfied that the arrest was warranted and that the case diary supports the necessity of detention.",
            "This judgment protects against misuse of domestic violence/cruelty provisions while maintaining protection for genuine victims. It applies to all offences punishable with up to 7 years imprisonment.",
        ],
    },
    {
        "case_name": "Rajesh Sharma v. State of Uttar Pradesh",
        "citation": "(2017) 8 SCC 446",
        "year": 2017,
        "domain": "criminal",
        "situation_types": ["domestic_violence"],
        "applicable_sections": ["IPC Section 498A", "BNS Section 85"],
        "holdings": [
            "The Supreme Court established a Family Welfare Committee in every district to examine complaints under Section 498A IPC (now BNS Section 85) before arrest or chargesheet.",
            "No arrest should normally be effected until the report of the Family Welfare Committee, unless the offence involves tangible physical injuries or death.",
            "This judgment aims to prevent misuse of matrimonial cruelty provisions while balancing the rights of genuinely aggrieved women. Courts must examine each case on its own merits.",
        ],
    },
    # -----------------------------------------------------------------------
    # CRIMINAL — Sexual Offences
    # -----------------------------------------------------------------------
    {
        "case_name": "Vishaka v. State of Rajasthan",
        "citation": "AIR 1997 SC 3011",
        "year": 1997,
        "domain": "criminal",
        "situation_types": ["sexual_offence"],
        "applicable_sections": ["Article 14", "Article 15", "Article 21", "POSH Act 2013"],
        "holdings": [
            "The Supreme Court laid down binding guidelines to prevent sexual harassment at the workplace, known as the 'Vishaka Guidelines', which remained in force until the POSH Act, 2013 was enacted.",
            "Sexual harassment includes unwelcome sexually determined behaviour such as physical contact, demand or request for sexual favours, sexually coloured remarks, showing pornography, or any unwelcome physical, verbal, or non-verbal conduct of a sexual nature.",
            "Every employer is obligated to constitute an Internal Complaints Committee (ICC) to address sexual harassment complaints. Failure to do so is a violation of fundamental rights under Articles 14, 15, and 21.",
            "The guidelines mandated preventive steps, complaint mechanisms, and disciplinary action, forming the basis for the comprehensive Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Act, 2013.",
        ],
    },
    {
        "case_name": "Tukaram v. State of Maharashtra (Mathura Rape Case)",
        "citation": "AIR 1979 SC 185",
        "year": 1979,
        "domain": "criminal",
        "situation_types": ["sexual_offence"],
        "applicable_sections": ["IPC Section 375", "BNS Section 63"],
        "holdings": [
            "This controversial acquittal in a custodial rape case led to massive public outcry and landmark amendments to rape laws in India.",
            "The aftermath of this judgment led to the Criminal Law Amendment Act 1983 which added Section 114A to the Indian Evidence Act — shifting the burden of proof to the accused in cases of custodial rape.",
            "The case established that consent obtained under fear or coercion is not valid consent. This principle is now codified in BNS Section 63 which defines rape and its exceptions.",
        ],
    },
    # -----------------------------------------------------------------------
    # CRIMINAL — General (Bail, Arrest, FIR)
    # -----------------------------------------------------------------------
    {
        "case_name": "Lalita Kumari v. Government of Uttar Pradesh",
        "citation": "(2014) 2 SCC 1",
        "year": 2014,
        "domain": "criminal",
        "situation_types": ["criminal_general"],
        "applicable_sections": ["CrPC Section 154", "BNSS Section 173"],
        "holdings": [
            "The Supreme Court held that registration of FIR is MANDATORY under Section 154 CrPC (now BNSS Section 173) when information discloses commission of a cognizable offence. The police officer cannot refuse to register the FIR.",
            "If the information does not disclose a cognizable offence but indicates the necessity for an inquiry, a preliminary inquiry may be conducted only to ascertain whether cognizable offence is disclosed or not.",
            "The scope of preliminary inquiry is limited and must be completed within 7 days. The police officer must record reasons in writing for not registering an FIR.",
            "This judgment ensures citizens' right to have their complaints registered. If police refuse, one can approach the Superintendent of Police or the Magistrate under BNSS Section 175(3).",
        ],
    },
    {
        "case_name": "Satender Kumar Antil v. CBI",
        "citation": "(2022) 10 SCC 51",
        "year": 2022,
        "domain": "criminal",
        "situation_types": ["criminal_general"],
        "applicable_sections": ["CrPC Section 436A", "BNSS Section 479"],
        "holdings": [
            "The Supreme Court laid down comprehensive bail guidelines directing courts to avoid unnecessary incarceration. Bail is the rule and jail is the exception.",
            "For offences punishable up to 3 years: notice to the accused is sufficient; custodial detention should normally be avoided.",
            "For offences punishable with 3-7 years: bail on personal bond should be the norm unless specific reasons recorded.",
            "Under-trial prisoners who have served half the maximum sentence must be released on bail under Section 436A CrPC (now BNSS Section 479).",
        ],
    },
    # -----------------------------------------------------------------------
    # CONSTITUTIONAL — Fundamental Rights
    # -----------------------------------------------------------------------
    {
        "case_name": "Maneka Gandhi v. Union of India",
        "citation": "AIR 1978 SC 597",
        "year": 1978,
        "domain": "constitutional",
        "situation_types": ["fundamental_rights"],
        "applicable_sections": ["Article 14", "Article 19", "Article 21"],
        "holdings": [
            "The Supreme Court expanded the scope of Article 21 (Right to Life) beyond mere animal existence to include the right to live with human dignity. Any procedure that deprives a person of life or liberty must be just, fair, and reasonable.",
            "Articles 14, 19, and 21 are not mutually exclusive but form a 'golden triangle' of fundamental rights. Any law affecting personal liberty under Article 21 must also satisfy the requirements of Article 14 (equality) and Article 19 (freedoms).",
            "The right to go abroad is part of personal liberty under Article 21. The government cannot impound a passport without following a fair and reasonable procedure.",
            "This landmark judgment transformed constitutional interpretation in India, establishing that fundamental rights must be interpreted broadly and liberally in favour of the citizen.",
        ],
    },
    {
        "case_name": "Kesavananda Bharati v. State of Kerala",
        "citation": "AIR 1973 SC 1461",
        "year": 1973,
        "domain": "constitutional",
        "situation_types": ["fundamental_rights"],
        "applicable_sections": ["Article 368", "Article 13"],
        "holdings": [
            "The Supreme Court established the 'Basic Structure Doctrine' — Parliament has the power to amend the Constitution under Article 368, but it cannot alter the basic structure or framework of the Constitution.",
            "The basic structure includes: supremacy of the Constitution, republican and democratic form of government, secular character, separation of powers, federal character, fundamental rights, and judicial review.",
            "This is one of the most important judgments in Indian constitutional history. It limits the amending power of Parliament and ensures that core constitutional values cannot be destroyed.",
        ],
    },
    {
        "case_name": "K.S. Puttaswamy v. Union of India",
        "citation": "(2017) 10 SCC 1",
        "year": 2017,
        "domain": "constitutional",
        "situation_types": ["fundamental_rights"],
        "applicable_sections": ["Article 21"],
        "holdings": [
            "The Supreme Court unanimously held that the Right to Privacy is a fundamental right protected under Article 21 of the Constitution. It overruled the earlier judgments in M.P. Sharma and Kharak Singh.",
            "Privacy includes: bodily privacy, informational privacy, and privacy of choice (including personal decisions about family, marriage, procreation, and sexual orientation).",
            "Any restriction on the right to privacy must satisfy three tests: legality (backed by law), legitimate aim (valid state interest), and proportionality (means must be proportionate to the objective).",
            "This judgment has far-reaching implications for data protection, surveillance, Aadhaar, and individual autonomy in India.",
        ],
    },
    # -----------------------------------------------------------------------
    # CONSTITUTIONAL — RTI / Transparency
    # -----------------------------------------------------------------------
    {
        "case_name": "CBSE v. Aditya Bandopadhyay",
        "citation": "(2011) 8 SCC 497",
        "year": 2011,
        "domain": "constitutional",
        "situation_types": ["rti_denial"],
        "applicable_sections": ["RTI Act 2005 Section 8", "Article 19(1)(a)"],
        "holdings": [
            "The Supreme Court held that the RTI Act provides access to all information that is available and existing. The PIO (Public Information Officer) is not required to create or collect new information that is not part of their records.",
            "Examined answer sheets of examinations can be provided under RTI as they form part of the records. However, RTI should not be used to seek information that would amount to evaluation of opinion.",
            "The Court emphasized that RTI is a powerful tool for transparency and accountability but must not be misused to create an unreasonable burden on public authorities. Genuine RTI seekers must not be deterred.",
        ],
    },
    {
        "case_name": "Namit Sharma v. Union of India",
        "citation": "(2013) 1 SCC 745",
        "year": 2013,
        "domain": "constitutional",
        "situation_types": ["rti_denial"],
        "applicable_sections": ["RTI Act 2005 Section 12", "RTI Act 2005 Section 15"],
        "holdings": [
            "The Supreme Court initially held that only persons with judicial experience should be appointed as Information Commissioners. However, this was modified on review.",
            "On review, the Court clarified that the RTI Act does not mandate judicial members and the original composition of Information Commissions should be maintained as per the Act. The government must ensure timely appointments to avoid vacancy-related delays.",
            "The judgment highlighted the importance of a functional and efficient Information Commission infrastructure for effective implementation of the RTI Act.",
        ],
    },
    # -----------------------------------------------------------------------
    # CONSUMER — Medical Negligence / Service Deficiency
    # -----------------------------------------------------------------------
    {
        "case_name": "Indian Medical Association v. V.P. Shantha",
        "citation": "(1995) 6 SCC 651",
        "year": 1995,
        "domain": "consumer",
        "situation_types": ["service_deficiency"],
        "applicable_sections": ["Consumer Protection Act 1986 Section 2(1)(o)", "Consumer Protection Act 2019 Section 2(42)"],
        "holdings": [
            "The Supreme Court held that medical practitioners provide 'service' within the meaning of the Consumer Protection Act. Patients are 'consumers' and can approach consumer forums for medical negligence.",
            "Services rendered free of charge (at government hospitals) where no charges are collected from any person are outside the purview of the Consumer Protection Act.",
            "However, services at government hospitals where charges are collected from patients who can afford to pay are covered under the Act. Similarly, private hospitals providing free treatment to some patients do not cease being 'service providers'.",
            "This landmark judgment opened up the healthcare sector to consumer protection law and established that medical negligence can be adjudicated by consumer forums, which are faster and cheaper than civil courts.",
        ],
    },
    {
        "case_name": "Jacob Mathew v. State of Punjab",
        "citation": "(2005) 6 SCC 1",
        "year": 2005,
        "domain": "consumer",
        "situation_types": ["service_deficiency"],
        "applicable_sections": ["IPC Section 304A", "BNS Section 106"],
        "holdings": [
            "The Supreme Court laid down guidelines for prosecuting doctors for negligence. A simple lack of care or an error of judgment does not amount to criminal negligence.",
            "For criminal liability in medical negligence, the degree of negligence must be 'gross' — meaning a deviation from the standard of a reasonably competent medical practitioner that shows reckless disregard for patient safety.",
            "The Court held that a private complaint for criminal negligence against a doctor cannot be entertained without prima facie evidence of gross negligence, preferably supported by an expert medical opinion.",
        ],
    },
    # -----------------------------------------------------------------------
    # CONSUMER — E-Commerce / Product Liability
    # -----------------------------------------------------------------------
    {
        "case_name": "Ambrish Kumar Shukla v. Ferrous Infrastructure Pvt. Ltd.",
        "citation": "2017 SCC OnLine NCDRC 1118",
        "year": 2017,
        "domain": "consumer",
        "situation_types": ["defective_product", "service_deficiency"],
        "applicable_sections": ["Consumer Protection Act 2019 Section 2(6)", "Consumer Protection Act 2019 Section 2(11)"],
        "holdings": [
            "The National Consumer Disputes Redressal Commission held that a real estate builder's failure to deliver possession of a flat on time constitutes 'deficiency in service' under the Consumer Protection Act.",
            "Allottees/home buyers are 'consumers' under the Act and entitled to refund with interest for delayed or non-delivery.",
            "The builder cannot take shelter behind 'force majeure' clauses in the agreement if the delay is attributable to the builder's own actions or negligence.",
        ],
    },
    # -----------------------------------------------------------------------
    # FAMILY — Maintenance / Matrimonial
    # -----------------------------------------------------------------------
    {
        "case_name": "Rajnesh v. Neha",
        "citation": "(2021) 2 SCC 324",
        "year": 2021,
        "domain": "family",
        "situation_types": ["maintenance", "divorce"],
        "applicable_sections": ["CrPC Section 125", "BNSS Section 144", "Hindu Marriage Act Section 24"],
        "holdings": [
            "The Supreme Court laid down comprehensive guidelines for maintenance cases. Both parties must file a sworn affidavit of income and assets (mandatory disclosure) with the very first application for maintenance.",
            "The Court mandated that overlapping maintenance proceedings should be considered together to avoid double payment. Maintenance awarded by one court should be adjusted against maintenance in another proceeding.",
            "Criteria for determining maintenance amount: status of the parties, reasonable needs of the wife and children, whether the applicant is educated and professionally qualified, whether the applicant has independent income, number of dependents, and the husband's actual income (not just disclosed income).",
            "Interim maintenance should be awarded within 60 days of service of notice on the respondent. The date of filing should be the effective date from which maintenance becomes payable.",
        ],
    },
    {
        "case_name": "Shilpa Sailesh v. Varun Sreenivasan",
        "citation": "(2023) 2 SCC 453",
        "year": 2023,
        "domain": "family",
        "situation_types": ["divorce"],
        "applicable_sections": ["Hindu Marriage Act Section 13B", "Article 142"],
        "holdings": [
            "The Supreme Court held that it can exercise its extraordinary power under Article 142 to grant divorce by mutual consent without the mandatory 6-month cooling-off period, where the marriage has irretrievably broken down.",
            "The Court also held it can dissolve a marriage even if only one party seeks divorce, if the Court is satisfied that the marriage has irretrievably broken down and there is no possibility of reconciliation.",
            "This judgment significantly impacts divorce law in India, providing relief in cases where prolonged litigation causes more harm than the dissolution itself.",
        ],
    },
    {
        "case_name": "Shamima Farooqui v. Shahid Khan",
        "citation": "(2015) 5 SCC 705",
        "year": 2015,
        "domain": "family",
        "situation_types": ["maintenance"],
        "applicable_sections": ["CrPC Section 125", "BNSS Section 144"],
        "holdings": [
            "The Supreme Court held that even a Muslim woman who has been divorced can claim maintenance under Section 125 CrPC (now BNSS Section 144) if she has not remarried and is unable to maintain herself.",
            "The obligation to pay maintenance to a divorced wife continues beyond the iddat period if she is unable to maintain herself and has not remarried.",
            "The husband's liability under Section 125 CrPC is not affected by the Muslim Women (Protection of Rights on Divorce) Act, 1986.",
        ],
    },
    # -----------------------------------------------------------------------
    # LABOUR — Wrongful Termination / Workers Rights
    # -----------------------------------------------------------------------
    {
        "case_name": "Workmen of Dimakuchi Tea Estate v. Dimakuchi Tea Estate",
        "citation": "(1958) 1 LLJ 500",
        "year": 1958,
        "domain": "labour",
        "situation_types": ["wrongful_termination"],
        "applicable_sections": ["Industrial Disputes Act Section 25F"],
        "holdings": [
            "The Supreme Court established that retrenchment of a workman can only be valid if the employer complies with three conditions under Section 25F of the Industrial Disputes Act: one month's written notice or wages in lieu thereof, retrenchment compensation at 15 days' wages for every completed year of service, and notice to the appropriate government.",
            "Non-compliance with Section 25F renders the retrenchment void ab initio (void from the beginning), and the workman is entitled to reinstatement with full back wages.",
        ],
    },
    {
        "case_name": "Secretary, State of Karnataka v. Umadevi",
        "citation": "(2006) 4 SCC 1",
        "year": 2006,
        "domain": "labour",
        "situation_types": ["wrongful_termination"],
        "applicable_sections": ["Article 14", "Article 16"],
        "holdings": [
            "The Supreme Court held that regularisation of temporary, daily-wage, or contractual employees who were not appointed through a proper selection process cannot be claimed as a matter of right under Articles 14 and 16.",
            "The Court directed that all public appointments must be made through a proper selection process based on merit. Ad hoc employees can only be regularised in exceptional circumstances.",
            "However, the judgment also provided that existing irregularly appointed employees who had served for 10+ years should be considered for regularisation as a one-time measure.",
        ],
    },
    {
        "case_name": "PUDR v. Union of India (Asiad Workers Case)",
        "citation": "AIR 1982 SC 1473",
        "year": 1982,
        "domain": "labour",
        "situation_types": ["unpaid_wages"],
        "applicable_sections": ["Article 23", "Article 24", "Minimum Wages Act 1948"],
        "holdings": [
            "The Supreme Court held that forcing a person to work for less than the minimum wage amounts to 'forced labour' prohibited under Article 23 of the Constitution.",
            "Every worker is entitled to receive at least the minimum wage. Any employment at less than minimum wage is exploitative and violates fundamental rights.",
            "This judgment is the basis for enforcing minimum wage laws. If your employer pays less than the state-notified minimum wage, it is a constitutional violation and can be challenged.",
        ],
    },
    # -----------------------------------------------------------------------
    # PROPERTY — Tenant / Succession
    # -----------------------------------------------------------------------
    {
        "case_name": "Vineeta Sharma v. Rakesh Sharma",
        "citation": "(2020) 9 SCC 1",
        "year": 2020,
        "domain": "property",
        "situation_types": ["land_dispute"],
        "applicable_sections": ["Hindu Succession Act Section 6"],
        "holdings": [
            "The Supreme Court held that daughters have equal coparcenary rights in Hindu joint family property by birth, irrespective of whether the father was alive on 09.09.2005 (when the Hindu Succession Amendment Act came into force) or not.",
            "The right of a daughter in coparcenary property is by birth and not dependent on whether the father was alive on the date of the amendment. Daughters have the same rights as sons in ancestral property.",
            "This judgment settled the conflicting views and established unambiguously that daughters' coparcenary rights are retrospective and conferred by birth, ensuring gender equality in property inheritance.",
        ],
    },
    {
        "case_name": "Raghunath Rai Bareja v. Punjab National Bank",
        "citation": "(2007) 2 SCC 230",
        "year": 2007,
        "domain": "property",
        "situation_types": ["tenant_rights"],
        "applicable_sections": ["Transfer of Property Act Section 106"],
        "holdings": [
            "The Supreme Court clarified that a tenant holding over after the expiry of the lease is not a trespasser but a tenant at sufferance. The landlord must follow due process of law for eviction.",
            "The notice for termination of tenancy must strictly comply with Section 106 of the Transfer of Property Act — 15 days' notice ending with the month of tenancy for monthly tenancy, and 6 months for year-to-year tenancy.",
            "The landlord cannot use self-help to evict a tenant — forcible eviction without court order is illegal and can attract criminal prosecution.",
        ],
    },
    # -----------------------------------------------------------------------
    # CRIMINAL — Mob Violence / Hate Crimes
    # -----------------------------------------------------------------------
    {
        "case_name": "Tehseen S. Poonawalla v. Union of India",
        "citation": "(2018) 9 SCC 501",
        "year": 2018,
        "domain": "criminal",
        "situation_types": ["assault", "criminal_general"],
        "applicable_sections": ["Article 14", "Article 15", "Article 21"],
        "holdings": [
            "The Supreme Court issued comprehensive directions to prevent mob violence and lynching. The Court held that the State has an affirmative obligation to protect the life and liberty of every citizen under Article 21.",
            "Directions included: designated senior police officer in each district as Nodal Officer for mob violence prevention, FIR to be registered immediately, compensation scheme for victims, fast-track trial for mob violence cases.",
            "The Court emphasized that vigilantism and mob justice cannot be condoned. No citizen has the right to take law into their own hands, and the State must act decisively to prevent and punish such acts.",
        ],
    },
    # -----------------------------------------------------------------------
    # CONSTITUTIONAL — Equality / Anti-Discrimination
    # -----------------------------------------------------------------------
    {
        "case_name": "Navtej Singh Johar v. Union of India",
        "citation": "(2018) 10 SCC 1",
        "year": 2018,
        "domain": "constitutional",
        "situation_types": ["fundamental_rights"],
        "applicable_sections": ["Article 14", "Article 15", "Article 19", "Article 21"],
        "holdings": [
            "The Supreme Court decriminalized consensual sexual conduct between adults of the same sex by reading down Section 377 of IPC. This expands Article 14 (equality), Article 15 (non-discrimination), Article 19 (freedom of expression), and Article 21 (dignity and privacy).",
            "The Court held that sexual orientation is an intrinsic aspect of personal identity and dignity. Criminalizing consensual sexual conduct violates the right to privacy, equality, and freedom of expression.",
            "The judgment reinforced that constitutional morality must prevail over social morality. The rights of minorities cannot be subjected to the will of the majority.",
        ],
    },
    {
        "case_name": "NALSA v. Union of India",
        "citation": "(2014) 5 SCC 438",
        "year": 2014,
        "domain": "constitutional",
        "situation_types": ["fundamental_rights"],
        "applicable_sections": ["Article 14", "Article 15", "Article 16", "Article 21"],
        "holdings": [
            "The Supreme Court recognized transgender persons as the 'third gender' and affirmed their fundamental rights under Articles 14, 15, 16, and 21 of the Constitution.",
            "Transgender persons have the right to self-identify their gender. They are entitled to reservation in education and employment as they constitute a socially and educationally backward class.",
            "The judgment directed the Centre and State governments to treat transgender persons as socially and educationally backward classes and extend all kinds of reservation in admissions and public appointments.",
        ],
    },
]

print(f"📚 {len(JUDGMENTS)} landmark judgments defined with "
      f"{sum(len(j['holdings']) for j in JUDGMENTS)} total holdings")


## 2. Build chunks from holdings

In [0]:
import hashlib, json
from datetime import datetime

rows = []
for j in JUDGMENTS:
    case_name = j["case_name"]
    citation = j["citation"]
    year = j["year"]
    domain = j["domain"]
    sections_str = ", ".join(j["applicable_sections"])

    for idx, holding in enumerate(j["holdings"]):
        chunk_id = hashlib.md5(f"{case_name}_{idx}".encode()).hexdigest()[:12]

        text = (
            f"[{case_name}, {citation} ({year})]\n\n"
            f"{holding}\n\n"
            f"Applicable sections: {sections_str}"
        )

        title = f"{case_name} — Holding {idx + 1}"

        rows.append({
            "chunk_id": f"SC_{chunk_id}",
            "title": title,
            "text": text,
            "source": citation,
            "doc_type": "sc_judgment",
            "domain": domain,
            "metadata_json": json.dumps({
                "case_name": case_name,
                "citation": citation,
                "year": year,
                "situation_types": j["situation_types"],
                "applicable_sections": j["applicable_sections"],
                "holding_index": idx,
            }),
        })

print(f"📄 {len(rows)} chunks created from {len(JUDGMENTS)} judgments")

## 3. Write to Delta Lake

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("title", StringType(), True),
    StructField("text", StringType(), True),
    StructField("source", StringType(), True),
    StructField("doc_type", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("metadata_json", StringType(), True),
])

df = spark.createDataFrame(rows, schema=schema)

TABLE_NAME = "workspace.default.legal_rag_corpus"

df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TABLE_NAME)

print(f"✅ Appended {len(rows)} SC judgment chunks to {TABLE_NAME}")

# Verify
total = spark.sql(f"SELECT count(*) as n FROM {TABLE_NAME}").collect()[0]["n"]
sc_count = spark.sql(f"SELECT count(*) as n FROM {TABLE_NAME} WHERE doc_type = 'sc_judgment'").collect()[0]["n"]

print(f"📊 Total corpus: {total} chunks ({sc_count} SC judgment chunks)")

## 4. Rebuild FAISS Index

After adding SC judgment chunks, rebuild the FAISS index by running:

```
notebooks/build_rag_index
```

This will re-embed all chunks (including new SC judgments) and create a fresh FAISS index.